In [ ]:
import platform
import subprocess
import sys
from pathlib import Path
import yaml

def check_apple_silicon():
    if platform.system() != "Darwin" or platform.machine() != "arm64":
        print(f"ERROR: MLX requires Apple Silicon macOS. Detected: {platform.system()} / {platform.machine()}")
        return False
    print(f"Platform OK: macOS on {platform.machine()}")
    return True

check_apple_silicon()

In [ ]:
def check_mlx_available():
    try:
        import mlx.core as mx
        import mlx_lm
        print("mlx-lm import OK.")
        return True
    except ImportError as e:
        print(f"ERROR: mlx / mlx-lm not installed. ({e})")
        return False

check_mlx_available()

In [ ]:
def load_config(config_path: str) -> dict:
    path = Path(config_path)
    if not path.exists():
        raise FileNotFoundError(f"Config file not found: {path}")
    with path.open() as f:
        return yaml.safe_load(f)

cfg = load_config("./lora_config.yaml")
print(cfg)

In [ ]:
def check_data_dir(data_dir: str):
    train_file = Path(data_dir) / "train.jsonl"
    valid_file = Path(data_dir) / "valid.jsonl"
    missing = [str(p) for p in (train_file, valid_file) if not p.exists()]
    if missing:
        print(f"ERROR: Missing data file(s): {missing}")
        return False
    n_train = sum(1 for _ in train_file.open())
    n_valid = sum(1 for _ in valid_file.open())
    print(f"Data OK: {n_train} train / {n_valid} valid examples in {data_dir}")
    return True

check_data_dir(cfg.get("data", "./data"))

In [ ]:
Path(cfg.get("adapter_path", "./adapters")).mkdir(parents=True, exist_ok=True)

print("=== Config summary ===")
for key in ("model", "iters", "batch_size", "learning_rate", "fine_tune_type", "num_layers"):
    print(f"  {key}: {cfg.get(key)}")
    

In [ ]:
import os
print(os.listdir("./adapters"))

In [ ]:
adapter_path = Path(cfg.get("adapter_path", "./adapters"))
resume_file = adapter_path / "adapters.safetensors"

cmd = [sys.executable, "-m", "mlx_lm", "lora", "--config", "./lora_config.yaml"]

if resume_file.exists():
    print(f"Found existing checkpoint: {resume_file} — resuming from it.")
    cmd += ["--resume-adapter-file", str(resume_file)]
else:
    print("No checkpoint found — starting fresh.")

print("Running:", " ".join(cmd))
result = subprocess.run(cmd)

if result.returncode != 0:
    print(f"Training failed with exit code {result.returncode}.")
else:
    print(f"Training complete. Adapter saved to: {adapter_path}")

In [ ]:
import os
print(os.listdir("./fused-model"))


In [33]:
from mlx_lm import load, generate

fused_model, fused_tokenizer = load("./fused-model")
print("Fused model loaded.")

Fused model loaded.


In [34]:
prompt = "Explain the difference between LoRA and full fine-tuning in two sentences."

chat = [{"role": "user", "content": prompt}]
formatted = fused_tokenizer.apply_chat_template(chat, add_generation_prompt=True, tokenize=False)

output = generate(fused_model, fused_tokenizer, prompt=formatted, max_tokens=200, verbose=True)
print(output)

LoRA (Low-Rank Approximation) is a technique that reduces the dimensionality of a model's weights by approximating the full weight matrix with a lower-rank approximation, resulting in a more interpretable and efficient model. In contrast, full fine-tuning involves training the model from scratch with the original weights, allowing it to adapt to the new task or dataset, but potentially leading to overfitting and requiring more data.
Prompt: 51 tokens, 12.790 tokens-per-sec
Generation: 88 tokens, 48.672 tokens-per-sec
Peak memory: 3.615 GB
LoRA (Low-Rank Approximation) is a technique that reduces the dimensionality of a model's weights by approximating the full weight matrix with a lower-rank approximation, resulting in a more interpretable and efficient model. In contrast, full fine-tuning involves training the model from scratch with the original weights, allowing it to adapt to the new task or dataset, but potentially leading to overfitting and requiring more data.
